# KnottedGraph vs Topoly: certified Dobrynin–Vesnin $\Theta(n)$ validation

This notebook contains **only** the certified Dobrynin–Vesnin $\Theta(n)$ benchmark. It measures the public KnottedGraph Yamada API and Topoly on the same canonical diagrams for $n=0,\ldots,20$.

The published formula is used **only after the timed computation as an external correctness oracle**. The KnottedGraph production evaluator does not recognize the family and does not call the formula.

Reference: A. A. Dobrynin and A. Yu. Vesnin, *The Yamada polynomial for graphs, embedded knot-wise into three-dimensional space*, Vychisl. Sistemy **155** (1996), 37–86, Theorem 2.

$$
R(\Theta(n))(A)=
(A^2+A+1+A^{-1}+A^{-2})A^n
-(A+A^{-1})A^{-2n}
-(A^2+1+A^{-2})(-1)^nA^{-n}.
$$


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm

import knotted_graph
from knotted_graph.invariants.yamada.native import native_available, native_import_error


def find_repo_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'dev' / 'benchmark_dobrynin_vesnin_theta_family.py').exists():
            return candidate
    raise RuntimeError('Could not locate the KnottedGraph repository root.')


ROOT = find_repo_root()
DEV = ROOT / 'dev'
if str(DEV) not in sys.path:
    sys.path.insert(0, str(DEV))

print('Repository root:', ROOT)
print('KnottedGraph import:', Path(knotted_graph.__file__).resolve())
print('Native Yamada backend available:', native_available())
print('Native import error:', native_import_error())
print('Note: optimized structural dispatch is used even when the native extension is unavailable.')


In [ ]:
THETA_N_MIN = 0
THETA_N_MAX = 20
THETA_TIMEOUT_S = 60.0


In [ ]:
from benchmark_dobrynin_vesnin_theta_family import (
    PAPER as DV_PAPER,
    FORMULA as DV_FORMULA,
    compare_one as compare_dv_theta,
)

print(DV_PAPER)
print(DV_FORMULA)


## Public-API timing and correctness sweep

Each row independently constructs the canonical $\Theta(n)$ diagram, times KnottedGraph and Topoly, and then compares their outputs with Theorem 2. KnottedGraph timing therefore measures the same public `Yamada(...).compute(...)` route used by library users.


In [ ]:
dv_rows = []
for n in tqdm(
    range(THETA_N_MIN, THETA_N_MAX + 1),
    desc='Dobrynin–Vesnin Theta(n)',
):
    dv_rows.append(compare_dv_theta(n, THETA_TIMEOUT_S))

len(dv_rows)


In [ ]:
dv_df = pd.DataFrame(dv_rows)
columns = [
    'n',
    'crossings',
    'abstract_graph',
    'knottedgraph_s',
    'topoly_s',
    'topoly_over_knottedgraph',
    'knottedgraph_vs_published',
    'topoly_vs_published',
]
dv_df[[column for column in columns if column in dv_df.columns]]


## Exactness gate

The critical requirement is that every completed KnottedGraph calculation agrees term-for-term with Dobrynin–Vesnin Theorem 2. Topoly is evaluated separately; a Topoly disagreement or arithmetic error is reported rather than used as ground truth.


In [ ]:
assert len(dv_rows) == THETA_N_MAX - THETA_N_MIN + 1
assert all(row['knottedgraph_status'] == 'ok' for row in dv_rows)
assert all(row['knottedgraph_vs_published'] == 'PASS' for row in dv_rows)

topoly_nonpass = [
    {
        'n': row['n'],
        'status': row['topoly_status'],
        'vs_published': row.get('topoly_vs_published'),
        'error': row.get('topoly_error'),
    }
    for row in dv_rows
    if row.get('topoly_vs_published') != 'PASS'
]

print('KnottedGraph: PASS for every n =', THETA_N_MIN, '...', THETA_N_MAX)
print('Topoly non-PASS cases:')
pd.DataFrame(topoly_nonpass)


In [ ]:
timed = dv_df[
    dv_df['knottedgraph_s'].notna() & dv_df['topoly_s'].notna()
].copy()

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(timed['n'], timed['knottedgraph_s'], marker='o', label='KnottedGraph')
ax.plot(timed['n'], timed['topoly_s'], marker='o', label='Topoly')
ax.set_yscale('log')
ax.set_xlabel('Crossings n')
ax.set_ylabel('Time (s, log scale)')
ax.set_title('Dobrynin–Vesnin Theta(n): public Yamada timing')
ax.legend()
ax.grid(alpha=0.25)
plt.show()


In [ ]:
ratio = timed[timed['topoly_over_knottedgraph'].notna()]
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(ratio['n'], ratio['topoly_over_knottedgraph'], marker='o')
ax.axhline(1.0, linewidth=1)
ax.set_xlabel('Crossings n')
ax.set_ylabel('Topoly time / KnottedGraph time')
ax.set_title('Speed ratio (>1 means KnottedGraph is faster)')
ax.grid(alpha=0.25)
plt.show()


## Interpretation

This notebook intentionally makes no broad claim about all spatial graphs. It establishes one reproducible, analytically certified benchmark family. The Dobrynin–Vesnin formula is excluded from the timed KnottedGraph computation and is used only as the post-computation oracle.

For longer stress tests beyond $n=20$, use the same benchmark script locally rather than extending GitHub CI.
